# Практика 1 — Pipeline: TF-IDF + мета-признаки

**Модуль 03 · AI HUB**

---

## Что делаем в этой практике

Продолжаем работу с **SMS Spam** из модуля 2. Baseline — `TF-IDF + LogReg` — уже работает.  
Сейчас мы его улучшаем: добавляем мета-признаки (`len`, `caps_ratio`, `exclaim_ratio`, `digit_ratio`) прямо внутрь Pipeline через `ColumnTransformer`.

1. Добавляем мета-признаки к датасету
2. Строим `ColumnTransformer` — TF-IDF на тексте, `StandardScaler` на числах одновременно
3. Собираем Pipeline и сравниваем с baseline через `StratifiedKFold(scoring='f1')`

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Настройка окружения

In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Всё импортировано успешно.")

Всё импортировано успешно.


---

## Задание 1 — Загрузка данных

Загружаем тот же датасет, что и в модуле 2.

**Что нужно сделать:**
1. Загрузите `data` из `data/` (tab-separated, без заголовка, колонки `label`, `text`)
2. Создайте числовой таргет `is_spam`: `spam` → 1, `ham` → 0
3. Удалите строки с пустым `text`
4. Выведите `df.shape` и баланс классов



In [ ]:
#df = pd.read_csv()
#ТВОЙ КОД ТУТ

---

## Задание 2 — Мета-признаки

В EDA из Практики 2 модуля 2 мы уже видели: спам длиннее, написан КАПСОМ и набит цифрами. Теперь добавим эти наблюдения в модель как числовые признаки.

**Что нужно сделать:**

Добавьте в `df` четыре колонки:
- `len` — длина сообщения в символах
- `caps_ratio` — доля заглавных букв (от длины, `clip(lower=1)` чтобы не делить на 0)
- `exclaim_ratio` — доля символов `!`
- `digit_ratio` — доля цифр (`\d`)

После добавления выведите `.describe()` по этим четырём колонкам, сгруппированное по `is_spam`.

In [ ]:
#ТВОЙ КОД ТУТ

---

## Задание 3 — Baseline из M2

Прежде чем улучшать, зафиксируем точку сравнения.

**Что нужно сделать:**
1. Создайте `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
2. Соберите baseline Pipeline (как в M2):
   ```python
   baseline = Pipeline([
       ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3)),
       ('clf',   LogisticRegression(max_iter=2000, class_weight='balanced')),
   ])
   ```
3. Запустите `cross_val_score` с `scoring='f1'` на `df['text']` / `df['is_spam']`
4. Сохраните результат: `scores_baseline = cross_val_score(...)`
5. Выведите `mean ± std`


In [ ]:
#ТВОЙ КОД ТУТ

**Baseline F1(spam):**  
*(запишите здесь mean ± std — это ваша точка сравнения)*

---

## Задание 4 — `ColumnTransformer`: текст + мета

Теперь соберём препроцессор, который одновременно обрабатывает два типа признаков.

**Что нужно сделать:**

Напишите функцию `build_preprocessor() -> ColumnTransformer`, которая:
- На колонку `'text'` применяет `TfidfVectorizer(ngram_range=(1, 2), min_df=3)`
- На колонки `META_COLS` применяет `StandardScaler()`
- Объединяет через `ColumnTransformer`

```python
META_COLS = ['len', 'caps_ratio', 'exclaim_ratio', 'digit_ratio']
```


In [ ]:
#ТВОЙ КОД ТУТ


---

## Задание 5 — Сборка и обучение Pipeline

**Что нужно сделать:**
1. Напишите функцию `build_pipeline(clf) -> Pipeline`:
   - Шаг `'pre'` — `build_preprocessor()`
   - Шаг `'clf'` — переданный классификатор
2. Создайте `pipe = build_pipeline(LogisticRegression(max_iter=2000, class_weight='balanced'))`
3. Запустите `cross_val_score` с теми же `cv` и `scoring='f1'`, но теперь на `df[['text'] + META_COLS]`
4. Сохраните `scores_meta = cross_val_score(...)`
5. Выведите `mean ± std`

In [ ]:
#ТВОЙ КОД ТУТ

---

## Задание 6 — Сравнение с baseline

**Что нужно сделать:**
1. Создайте итоговую таблицу сравнения:

| Pipeline | F1(spam) mean | F1(spam) std | Δ vs baseline |
|----------|--------------|-------------|---------------|
| TF-IDF + LogReg (baseline M2) | ? | ? | — |
| TF-IDF + мета + LogReg | ? | ? | ? |

2. Постройте boxplot двух распределений F1 по фолдам (как в Практике 2 модуля 2)
3. Напишите вывод: дают ли мета-признаки прирост?

In [ ]:
#ТВОЙ КОД ТУТ

**Вывод:**  
*(напишите здесь: дают ли мета-признаки прирост, и как это объяснить)*

---

## Задание 7 — Проверка на новом сообщении

Pipeline должен принимать сырой текст — без предвычисленных мета-признаков.

**Проблема:** `cross_val_score` обучал Pipeline на `df[['text'] + META_COLS]`. Но при `predict` на новом сообщении мета-признаки нужно считать заново — вручную. Это неудобно.

**Что нужно сделать:**
1. Обучите финальный Pipeline на всём датасете: `pipe.fit(df[['text'] + META_COLS], df['is_spam'])`
2. Создайте функцию `predict_sms(text: str) -> dict`, которая:
   - Считает мета-признаки для одного SMS
   - Формирует `pd.DataFrame` с нужными колонками
   - Возвращает `{'prediction': 'spam'/'ham', 'proba': float}`
3. Проверьте на двух примерах:
   - `"WINNER!!! You have been selected for a $1000 prize. Call now!"`
   - `"Hey, are we still meeting at 6?"`

In [ ]:
#ТВОЙ КОД ТУТ

---

## Итог практики 1

Если все задания выполнены:
- Добавлены 4 мета-признака в датасет
- `ColumnTransformer` объединяет TF-IDF и `StandardScaler` в одном шаге
- Pipeline оценён через `StratifiedKFold(scoring='f1')` — честно, без утечек
- Есть сравнение с baseline M2

**Следующий шаг:** Практика 2 — прогоним 4 модели в едином Pipeline через GridSearchCV.